In [1]:
# !pip install codecarbon
# !pip install -e .

import pandas as pd
import pickle
from trust_library.core import TrustEvaluator
from trust_library.factsheet import (
    load_factsheet_default,
    load_factsheet,
    save_factsheet, 
    create_factsheet_interactive,
    create_factsheet,
)
from trust_library.fairness import statistical_parity_difference

import matplotlib.pyplot as plt
import seaborn as sns
import json

from trust_library.utils import to_json_safe
import plotly.express as px

In [2]:
# factsheet = create_factsheet_interactive()
# print(load_factsheet_default()) 
# fs = create_factsheet({
# "general": {
#     "target_column": "Target"
# },
# "fairness": {
#     "protected_feature": "Group",
#     "protected_values": [1],      # Valor que identifica al grupo desprotegido
#     "favorable_outcomes": [1]     # Valor que identifica el éxito
# }
# })
# save_factsheet(fs, "factsheet.json")

factsheet = load_factsheet_default() #load_factsheet("factsheet.json")

with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

train_loaded = pd.read_csv("train.csv")
test_loaded = pd.read_csv("test.csv")

evaluator = TrustEvaluator(
    model=loaded_model,
    train_data=train_loaded,
    test_data=test_loaded,
    factsheet=factsheet
)

In [3]:
# results = evaluator.evaluate()

# print("\n=== RESULTADOS FINAL ===")
# print(f"Global Trust Score: {results['trust_score']}")
# print(f"Pillar Scores:\n{json.dumps(results['pillar_score'], indent=4)}")
# print("\nDetalles (Scores brutos):\n" + json.dumps(results['details'], indent=4))

# print("\nPropiedades calculadas:\n" + json.dumps(to_json_safe(results['properties']), indent=4))
# print("\n=== EXPLICACIÓN DEL SCORE ===")
# print(json.dumps(results['explanation'], indent=4))

In [4]:
# y_pred = loaded_model.predict(
#     test_loaded.drop(
#         columns=[factsheet["general"]["target_column"]["value"]]
#     )
# )

# group_mask = (
#     test_loaded[factsheet["fairness"]["protected_feature"]["value"]]
#     == factsheet["fairness"]["protected_values"]["value"][0]
# )

y_pred = loaded_model.predict(test_loaded.drop("Target", axis=1))
group_mask = test_loaded["Group"] == 1

statistical_parity_difference(y_pred,group_mask)

{'value': 0.36741608418036453,
 'favored_ratio_protected': 0.5766483972762,
 'favored_ratio_unprotected': 0.20923231309583543,
 'n_protected': 6021,
 'n_unprotected': 5979,
 'n_protected_favored': 3472,
 'n_unprotected_favored': 1251}

In [5]:
subset = evaluator.evaluate(["fairness", "explainability", "accountability", "sustainability"], verbose=True)
# Imprimimos subset formateado
print("\n=== SUBSET DE PILLARS (Fairness, Privacy, Accountability y Sustainability) ===")
print(json.dumps(subset, indent=4))

Computing Fairness metrics...
Computing Explainability metrics...
Computing Accountability metrics...
Computing Sustainability metrics...

=== SUBSET DE PILLARS (Fairness, Privacy, Accountability y Sustainability) ===
{
    "trust_score": 2.4,
    "pillar_score": {
        "fairness": 3.1,
        "explainability": 3.5,
        "accountability": 1.8,
        "sustainability": 1.0
    },
    "details": {
        "fairness": {
            "underfitting": 5,
            "overfitting": 2,
            "class_balance": 1,
            "statistical_parity_difference": 1,
            "disparate_impact": 5,
            "equal_opportunity_difference": 3,
            "average_odds_difference": 4,
            "accuracy_parity": 5,
            "predictive_parity": 3,
            "treatment_equality": 2,
            "calibration_gap": 3,
            "well_calibration_error": 4,
            "generalized_entropy_index": 1,
            "theil_index": 4,
            "coefficient_of_variation": 1,
       

In [6]:
evaluator.plot_results()

In [7]:
evaluator.plot_radar()

In [8]:
# subset = evaluator.evaluate(["robustness"], verbose=True)
# # Imprimimos subset formateado
# print("\n=== SUBSET DE PILLARS (Robustness) ===")
# print(json.dumps(subset, indent=4))

In [10]:
with open("model_SVM.pkl", "rb") as f:
    loaded_model2 = pickle.load(f)
evaluator2 = TrustEvaluator(loaded_model2, train_loaded, test_loaded, factsheet)
subset2 = evaluator2.evaluate(["fairness", "accountability", "sustainability"], verbose=True)



Computing Fairness metrics...
Computing Accountability metrics...
Computing Sustainability metrics...


In [11]:

TrustEvaluator.compare_radar({
    "RandomForest": subset,
    "SVM": subset2
})

TrustEvaluator.compare_all_bars({
    "RandomForest": subset,
    "SVM": subset2
})